# FMCG Sales Intelligence — Integration Example

This tutorial demonstrates supported integration boundaries. It performs no training and never writes canonical artifacts.

In [2]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'examples':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'examples'))
from backend_integration import run_direct_usage  # noqa: E402

usage = run_direct_usage()
'Frozen usage results loaded'

C:\Users\Eclipse\PycharmProjects\FMCG-Sales-Intelligence\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape
C:\Users\Eclipse\PycharmProjects\FMCG-Sales-Intelligence\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape
C:\Users\Eclipse\PycharmProjects\FMCG-Sales-Intelligence\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape
C:\Users\Eclipse\PycharmProjects\FMCG-Sales-Intellig

'Frozen usage results loaded'

## Available capabilities

The domain pack declares available canonical contracts. Capability evaluation prevents a missing contract from being treated as a usable model input.

In [3]:
pd.DataFrame(usage['domain_capabilities'])[['capability', 'status', 'missing_or_invalid']]

,capability,status,missing_or_invalid
0,forecasting,AVAILABLE,[]
1,stockout_classification,UNAVAILABLE_MISSING_CONTRACT,"[deliveries, inventory_snapshots]"
2,stockout_survival,UNAVAILABLE_MISSING_CONTRACT,"[deliveries, inventory_snapshots]"
3,segmentation,AVAILABLE,[]
4,anomaly,AVAILABLE,[]
5,basket,UNAVAILABLE_MISSING_CONTRACT,"[order_items, orders]"
6,promotion,UNAVAILABLE_MISSING_CONTRACT,"[promotion_skus, promotion_stores, promotions]"
7,segment_cluster_membership_assignment,ACTIVE_EXPERIMENTAL,[]
8,supervised_segment_classification,BLOCKED_SCIENTIFICALLY,[]


## External data → canonical contract

`FileAdapter` applies the selected domain-pack mapping. `ContractRegistry` then checks required columns, nulls and business-key duplicates. A structural PASS is necessary but does not replace semantic data-quality review.

In [4]:
usage['external_data_contract']['normalized_rows'], usage['external_data_contract']['validation']

([{'observation_date': '2026-09-01',
   'store_id': 101,
   'sku_id': 501,
   'observed_units': 12.0}],
 {'contract_id': 'sales_daily',
  'contract_version': 'v1',
  'row_count': 1,
  'required_columns_found': ['observation_date',
   'observed_units',
   'sku_id',
   'store_id'],
  'missing_required_columns': [],
  'unexpected_columns': [],
  'null_failures': {},
  'duplicate_key_count': 0,
  'status': 'PASS'})

## Forecasting — ONLINE / BATCH SERVING

`ForecastService` accepts the frozen V1 feature contract and returns the predicted realized units in `(t,t+7d]`. Raw `sales_daily` rows must first pass feature construction.

In [5]:
pd.Series(usage['forecasting']['input'], name='forecast input')

lag_1                               0.0
lag_7                               4.0
lag_14                              4.0
lag_28                              0.0
sales_velocity_7d                    14
rolling_mean_7d                     2.0
rolling_std_7d                      2.0
scheduled_selling_price            3.83
store_id                             10
sku_id                               39
region_id                             2
brand_name                      Orchard
category_name                     Juice
store_type                  convenience
channel                    modern_trade
Name: forecast input, dtype: object

In [6]:
usage['forecasting']['result']

{'model': 'forecasting',
 'model_version': 'forecasting_v1',
 'prediction_timestamp': '2026-09-23T21:43:48.176784+00:00',
 'prediction_target': 'target_units_next_7d',
 'prediction_horizon': '(t,t+7d]',
 'predicted_units': 7.575987283169399,
 'nonnegative_clipping': True}

## Stockout Classification — ONLINE / BATCH SERVING

`StockoutService` returns a probability and the frozen validation-derived decision threshold. The score is not claimed to be strongly calibrated.

In [7]:
usage['stockout_classification']

{'input': {'stock_quantity': 71,
  'reserved_quantity': 0,
  'available_quantity': 71,
  'reorder_point': 73,
  'safety_stock': 26,
  'distance_to_reorder_point': -2,
  'replenishment_sum_7d': Decimal('136'),
  'stock_to_safety_ratio': 2.730769230769231,
  'sales_velocity_7d': 5.5625,
  'warehouse_id': 3,
  'sku_id': 6},
 'result': {'model': 'stockout_classification',
  'model_version': 'stockout_classification_v1',
  'prediction_timestamp': '2026-09-23T21:43:48.219638+00:00',
  'prediction_target': 'stockout occurrence within (t,t+7d]',
  'prediction_horizon': '(t,t+7d]',
  'stockout_probability': 0.022034461844789233,
  'decision_threshold': 0.7695666515458811,
  'risk_flag': False,
  'calibration_note': 'Score is not claimed to be strongly calibrated.'}}

## Store Segmentation — EXPLORATORY FROZEN ASSIGNMENT

The service applies the existing KMeans state without fitting. Its cluster is a pseudo-label with limited taxonomy stability and mandatory review.

In [8]:
usage['segmentation_membership']

{'input': {'store_id': 10,
  'revenue_30d': Decimal('6476.38'),
  'average_price_30d': 2.641111111111111,
  'promotion_unit_share_30d': 0.16483931947069944,
  'revenue_volatility_30d': 5.623794681589026},
 'result': {'store_id': 10,
  'cluster_id': 0,
  'profile_label': 'Lower-Volume Lower-Promotion',
  'centroid_distance': 1.0363314017379242,
  'segmentation_model_version': 'segmentation_v1',
  'feature_contract_version': 'segmentation_features_v1',
  'artifact_fingerprint': '61c1dd118d738d48dc32da9a75dfd7fd0009b4731117f9711be00874f99b0a41',
  'assignment_method': 'frozen_kmeans_predict',
  'scientific_status': 'exploratory',
  'taxonomy_stability': 'limited',
  'review_required': True,
  'label_semantics': 'PSEUDO_LABEL_NOT_GROUND_TRUTH'}}

## Anomaly Detection — OFFLINE HUMAN-REVIEW ANALYTICS

The product interface reads frozen review candidates. It does not expose a fake online anomaly predictor and does not treat candidates as confirmed events.

In [9]:
anomalies = usage['offline_frozen_outputs']['anomalies']
pd.DataFrame(anomalies['items'])[['event_date', 'store_id', 'sku_id', 'iforest_anomaly_score', 'review_status']]

,event_date,store_id,sku_id,iforest_anomaly_score,review_status
0,2024-02-23,16,1,0.071938,UNREVIEWED
1,2024-03-08,18,1,0.030573,UNREVIEWED


## Market Basket Analysis — OFFLINE ASSOCIATION ANALYTICS

Rules describe co-occurrence. They are not recommendations and do not establish causality.

In [10]:
rules = usage['offline_frozen_outputs']['basket-rules']
pd.DataFrame(rules['items'])[['antecedent', 'consequent', 'support', 'confidence', 'lift']]

,antecedent,consequent,support,confidence,lift
0,32,31,0.0525,0.283019,1.645459
1,31,32,0.0525,0.305233,1.645459


## Promotion Performance — OFFLINE DESCRIPTIVE ANALYTICS

The frozen output is a PRE/DURING/POST comparison. It is explicitly non-causal and does not claim incrementality or ROI.

In [11]:
promotions = usage['offline_frozen_outputs']['promotions']
pd.DataFrame(promotions['items'])[['promotion_id', 'pre_units_per_day', 'during_units_per_day', 'post_units_per_day', 'analysis_semantics']]

,promotion_id,pre_units_per_day,during_units_per_day,post_units_per_day,analysis_semantics
0,1,NaN,1.707986,1.836076,DESCRIPTIVE_NON_CAUSAL
1,2,1.707986,1.836076,1.928299,DESCRIPTIVE_NON_CAUSAL


## Stockout Survival — OFFLINE ANALYTICAL SCORING

The scientific API applies the frozen Cox PH bundle to an eligible feature row. Survival probabilities are horizon-specific event-free probabilities, not classification probabilities. No REST endpoint is declared.

In [12]:
usage['stockout_survival']

{'input': {'available_quantity': 71,
  'reorder_point': 73,
  'safety_stock': 26,
  'sales_velocity_7d': 5.5625,
  'replenishment_sum_7d': Decimal('136')},
 'result': {'risk_score': 1.6642923903582894,
  'survival_probability_day_3': 0.9862110940666529,
  'survival_probability_day_5': 0.9765297445407383,
  'survival_probability_day_7': 0.9669917604219234}}

## Integration summary

- **Serving:** forecasting, stockout classification and exploratory segment membership use frozen artifacts.
- **Offline scoring:** survival has a Python scientific interface but no REST route.
- **Frozen analytical reads:** anomaly candidates, basket rules and promotion summaries are available in Python and REST.
- **External onboarding:** map source columns, validate canonical contracts, evaluate capability availability, then run the reviewed task-dataset pipeline.
- **Not performed here:** model fitting, canonical artifact generation, DVC operations or MLflow runtime lookup.